In [2]:

# ============================================================
# RAILPULSE AI — CHRONOLOGICAL BASELINE MODELLING
# Dummy Regressor vs Linear Regression
# ============================================================

import os
import time
import gc
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

DATA_PATH = "../data/processed/railpulse_model_features.csv"
OUTPUT_DIR = "../outputs/tables"

os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 100_000

FEATURE_COLS = [
    "current_arr_delay",
    "distance_to_next",
    "current_distance",
    "current_sequence",
    "arrival_hour_sin",
    "arrival_hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "is_weekend"
]

TARGET_COL = "next_arr_delay"

USE_COLS = ["date"] + FEATURE_COLS + [TARGET_COL]

# ------------------------------------------------------------
# 2. LOAD DATA IN MEMORY-EFFICIENT CHUNKS
# ------------------------------------------------------------

print("=" * 65)
print("RAILPULSE AI — CHRONOLOGICAL MODELLING")
print("=" * 65)

X_train_parts = []
y_train_parts = []

X_test_parts = []
y_test_parts = []

print("\nLoading data in chunks...")

total_rows = 0

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=USE_COLS,
    chunksize=CHUNK_SIZE
):

    total_rows += len(chunk)

    # September 1–20 = training
    # September 21–30 = testing
    day = pd.to_datetime(
        chunk["date"],
        errors="coerce"
    ).dt.day

    train_mask = day <= 20
    test_mask = day >= 21

    # Convert predictors to float32 to save RAM
    X = chunk[FEATURE_COLS].astype(np.float32)

    y = pd.to_numeric(
        chunk[TARGET_COL],
        errors="coerce"
    ).astype(np.float32)

    if train_mask.any():
        X_train_parts.append(
            X.loc[train_mask].to_numpy(copy=True)
        )

        y_train_parts.append(
            y.loc[train_mask].to_numpy(copy=True)
        )

    if test_mask.any():
        X_test_parts.append(
            X.loc[test_mask].to_numpy(copy=True)
        )

        y_test_parts.append(
            y.loc[test_mask].to_numpy(copy=True)
        )

    if total_rows % 300_000 < CHUNK_SIZE:
        print(f"Processed {total_rows:,} rows")

# Combine chunks
X_train = np.concatenate(X_train_parts)
y_train = np.concatenate(y_train_parts)

X_test = np.concatenate(X_test_parts)
y_test = np.concatenate(y_test_parts)

# Release temporary lists
del X_train_parts, y_train_parts
del X_test_parts, y_test_parts

gc.collect()

print("\nDATA LOADING COMPLETE")
print("Training shape:", X_train.shape)
print("Testing shape :", X_test.shape)

print("Training rows:", f"{len(y_train):,}")
print("Testing rows :", f"{len(y_test):,}")

# Safety checks
assert X_train.shape[1] == len(FEATURE_COLS)
assert X_test.shape[1] == len(FEATURE_COLS)

assert not np.isnan(X_train).any()
assert not np.isnan(X_test).any()
assert not np.isnan(y_train).any()
assert not np.isnan(y_test).any()

print("\nNo missing values in modelling arrays.")

# ------------------------------------------------------------
# 3. DEFINE MODELS
# ------------------------------------------------------------

models = {

    "Dummy Regressor": DummyRegressor(
        strategy="mean"
    ),

    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])

}

# ------------------------------------------------------------
# 4. TRAIN AND EVALUATE
# ------------------------------------------------------------

results = []

for model_name, model in models.items():

    print("\n" + "=" * 65)
    print("MODEL:", model_name)
    print("=" * 65)

    start_time = time.time()

    # Train
    model.fit(X_train, y_train)

    # Predict
    predictions = model.predict(X_test)

    # Evaluation
    mae = mean_absolute_error(
        y_test,
        predictions
    )

    mse = mean_squared_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test,
        predictions
    )

    elapsed = time.time() - start_time

    results.append({
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "Training_Time_Seconds": elapsed
    })

    print(f"MAE  : {mae:.4f}")
    print(f"MSE  : {mse:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")
    print(f"Time : {elapsed:.2f} seconds")

    del predictions
    gc.collect()

# ------------------------------------------------------------
# 5. COMPARATIVE RESULTS
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="MAE"
).reset_index(drop=True)

print("\n" + "=" * 65)
print("BASELINE MODEL COMPARISON")
print("=" * 65)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 6. SAVE RESULTS
# ------------------------------------------------------------

results_path = os.path.join(
    OUTPUT_DIR,
    "chronological_baseline_results.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

print("\nResults saved to:")
print(results_path)

print("\nBASELINE MODELLING COMPLETED SUCCESSFULLY.")


RAILPULSE AI — CHRONOLOGICAL MODELLING

Loading data in chunks...
Processed 300,000 rows
Processed 600,000 rows
Processed 900,000 rows

DATA LOADING COMPLETE
Training shape: (730719, 9)
Testing shape : (370724, 9)
Training rows: 730,719
Testing rows : 370,724

No missing values in modelling arrays.

MODEL: Dummy Regressor
MAE  : 34.8262
MSE  : 4576.7905
RMSE : 67.6520
R2   : -0.0033
Time : 0.04 seconds

MODEL: Linear Regression
MAE  : 9.7216
MSE  : 816.6270
RMSE : 28.5767
R2   : 0.8210
Time : 0.48 seconds

BASELINE MODEL COMPARISON
            Model     MAE       MSE    RMSE      R2  Training_Time_Seconds
Linear Regression  9.7216  816.6270 28.5767  0.8210                 0.4799
  Dummy Regressor 34.8262 4576.7905 67.6520 -0.0033                 0.0438

Results saved to:
../outputs/tables\chronological_baseline_results.csv

BASELINE MODELLING COMPLETED SUCCESSFULLY.


In [3]:

# ============================================================
# RAILPULSE AI — ADVANCED BASELINES AND NONLINEAR MODELS
# ============================================================

import os
import gc
import time
import numpy as np
import pandas as pd

from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

OUTPUT_DIR = "../outputs/tables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42

# ------------------------------------------------------------
# 2. EVALUATION FUNCTION
# ------------------------------------------------------------

advanced_results = []

def evaluate_model(name, predictions, elapsed):

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)

    result = {
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "Training_Time_Seconds": elapsed
    }

    advanced_results.append(result)

    print(f"\n{name}")
    print("-" * 50)
    print(f"MAE  : {mae:.4f}")
    print(f"MSE  : {mse:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")
    print(f"Time : {elapsed:.2f} seconds")


# ------------------------------------------------------------
# 3. PERSISTENCE BASELINE
# Predict next delay = current delay
# ------------------------------------------------------------

print("=" * 65)
print("MODEL 1 — PERSISTENCE BASELINE")
print("=" * 65)

start = time.time()

current_delay_index = FEATURE_COLS.index(
    "current_arr_delay"
)

persistence_predictions = X_test[
    :, current_delay_index
].astype(np.float64)

evaluate_model(
    "Persistence Baseline",
    persistence_predictions,
    time.time() - start
)

del persistence_predictions
gc.collect()


# ------------------------------------------------------------
# 4. RANDOM FOREST
# Memory-controlled sample of training data
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MODEL 2 — RANDOM FOREST")
print("=" * 65)

rng = np.random.default_rng(RANDOM_STATE)

RF_SAMPLE_SIZE = min(
    250_000,
    len(X_train)
)

rf_indices = rng.choice(
    len(X_train),
    size=RF_SAMPLE_SIZE,
    replace=False
)

X_rf = X_train[rf_indices]
y_rf = y_train[rf_indices]

print(f"Random Forest training sample: {len(X_rf):,}")

rf_model = RandomForestRegressor(
    n_estimators=60,
    max_depth=16,
    min_samples_leaf=5,
    max_features=0.8,
    max_samples=0.8,
    n_jobs=2,
    random_state=RANDOM_STATE
)

start = time.time()

rf_model.fit(X_rf, y_rf)

rf_predictions = rf_model.predict(X_test)

rf_time = time.time() - start

evaluate_model(
    "Random Forest",
    rf_predictions,
    rf_time
)

del X_rf, y_rf, rf_indices, rf_predictions
del rf_model
gc.collect()


# ------------------------------------------------------------
# 5. HISTOGRAM GRADIENT BOOSTING
# Uses the full training dataset
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MODEL 3 — HISTOGRAM GRADIENT BOOSTING")
print("=" * 65)

hgb_model = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    min_samples_leaf=30,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=15,
    random_state=RANDOM_STATE
)

start = time.time()

hgb_model.fit(X_train, y_train)

hgb_predictions = hgb_model.predict(X_test)

hgb_time = time.time() - start

evaluate_model(
    "HistGradientBoosting",
    hgb_predictions,
    hgb_time
)

del hgb_predictions
gc.collect()


# ------------------------------------------------------------
# 6. COMBINE RESULTS
# ------------------------------------------------------------

advanced_df = pd.DataFrame(advanced_results)

# Add the earlier baseline results if available
if "results_df" in globals():

    combined_df = pd.concat(
        [results_df, advanced_df],
        ignore_index=True
    )

else:
    combined_df = advanced_df.copy()

combined_df = combined_df.drop_duplicates(
    subset=["Model"],
    keep="last"
)

combined_df = combined_df.sort_values(
    by="MAE"
).reset_index(drop=True)

print("\n" + "=" * 75)
print("COMPLETE CHRONOLOGICAL MODEL COMPARISON")
print("=" * 75)

print(
    combined_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------

result_path = os.path.join(
    OUTPUT_DIR,
    "chronological_model_comparison.csv"
)

combined_df.to_csv(
    result_path,
    index=False
)

print("\nResults saved to:")
print(result_path)

print("\nADVANCED MODELLING COMPLETED.")


MODEL 1 — PERSISTENCE BASELINE

Persistence Baseline
--------------------------------------------------
MAE  : 8.9924
MSE  : 842.5006
RMSE : 29.0259
R2   : 0.8153
Time : 0.00 seconds

MODEL 2 — RANDOM FOREST
Random Forest training sample: 250,000

Random Forest
--------------------------------------------------
MAE  : 10.1862
MSE  : 803.2561
RMSE : 28.3418
R2   : 0.8239
Time : 46.48 seconds

MODEL 3 — HISTOGRAM GRADIENT BOOSTING

HistGradientBoosting
--------------------------------------------------
MAE  : 10.5644
MSE  : 911.8410
RMSE : 30.1967
R2   : 0.8001
Time : 16.02 seconds

COMPLETE CHRONOLOGICAL MODEL COMPARISON
               Model     MAE       MSE    RMSE      R2  Training_Time_Seconds
Persistence Baseline  8.9924  842.5006 29.0259  0.8153                 0.0014
   Linear Regression  9.7216  816.6270 28.5767  0.8210                 0.4799
       Random Forest 10.1862  803.2561 28.3418  0.8239                46.4808
HistGradientBoosting 10.5644  911.8410 30.1967  0.8001      

In [4]:

# ============================================================
# RAILPULSE AI — UNSEEN-TRAIN GENERALIZATION EXPERIMENT
# ============================================================

import os
import gc
import time
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

DATA_PATH = "../data/processed/railpulse_model_features.csv"

MANIFEST_PATH = (
    "../outputs/tables/unseen_train_split_manifest.csv"
)

OUTPUT_DIR = "../outputs/tables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 100_000
RANDOM_STATE = 42

FEATURE_COLS = [
    "current_arr_delay",
    "distance_to_next",
    "current_distance",
    "current_sequence",
    "arrival_hour_sin",
    "arrival_hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "is_weekend"
]

TARGET_COL = "next_arr_delay"

# ------------------------------------------------------------
# 2. LOAD TRAIN MANIFEST
# ------------------------------------------------------------

print("=" * 65)
print("RAILPULSE AI — UNSEEN-TRAIN EXPERIMENT")
print("=" * 65)

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={"train": str}
)

train_ids = set(
    manifest.loc[
        manifest["split"] == "train",
        "train"
    ]
)

test_ids = set(
    manifest.loc[
        manifest["split"] == "test",
        "train"
    ]
)

assert train_ids.isdisjoint(test_ids)

print("Training trains:", len(train_ids))
print("Testing trains :", len(test_ids))
print("Train overlap  :", len(train_ids & test_ids))

# ------------------------------------------------------------
# 3. LOAD DATA USING THE MANIFEST
# ------------------------------------------------------------

X_train_parts = []
y_train_parts = []

X_test_parts = []
y_test_parts = []

total_rows = 0

print("\nLoading unseen-train split in chunks...")

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["train"] + FEATURE_COLS + [TARGET_COL],
    dtype={"train": str},
    chunksize=CHUNK_SIZE
):

    total_rows += len(chunk)

    train_mask = chunk["train"].isin(train_ids)
    test_mask = chunk["train"].isin(test_ids)

    X = chunk[FEATURE_COLS].to_numpy(
        dtype=np.float32,
        copy=True
    )

    y = chunk[TARGET_COL].to_numpy(
        dtype=np.float32,
        copy=True
    )

    if train_mask.any():
        X_train_parts.append(X[train_mask.to_numpy()])
        y_train_parts.append(y[train_mask.to_numpy()])

    if test_mask.any():
        X_test_parts.append(X[test_mask.to_numpy()])
        y_test_parts.append(y[test_mask.to_numpy()])

    if total_rows % 300_000 < CHUNK_SIZE:
        print(f"Processed {total_rows:,} rows")

X_train_unseen = np.concatenate(X_train_parts)
y_train_unseen = np.concatenate(y_train_parts)

X_test_unseen = np.concatenate(X_test_parts)
y_test_unseen = np.concatenate(y_test_parts)

del X_train_parts, y_train_parts
del X_test_parts, y_test_parts
gc.collect()

print("\nDATA LOADING COMPLETE")
print("Training shape:", X_train_unseen.shape)
print("Testing shape :", X_test_unseen.shape)

assert X_train_unseen.shape[1] == 9
assert X_test_unseen.shape[1] == 9

assert len(X_train_unseen) + len(X_test_unseen) == total_rows

print("Total rows:", f"{total_rows:,}")

# ------------------------------------------------------------
# 4. EVALUATION FUNCTION
# ------------------------------------------------------------

unseen_results = []

def evaluate_unseen(name, predictions, elapsed):

    mae = mean_absolute_error(
        y_test_unseen,
        predictions
    )

    mse = mean_squared_error(
        y_test_unseen,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test_unseen,
        predictions
    )

    unseen_results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "Training_Time_Seconds": elapsed
    })

    print(f"\n{name}")
    print("-" * 50)
    print(f"MAE  : {mae:.4f}")
    print(f"MSE  : {mse:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")
    print(f"Time : {elapsed:.2f} seconds")

# ------------------------------------------------------------
# 5. PERSISTENCE BASELINE
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MODEL 1 — PERSISTENCE BASELINE")
print("=" * 65)

start = time.time()

delay_index = FEATURE_COLS.index(
    "current_arr_delay"
)

persistence_predictions = X_test_unseen[
    :, delay_index
]

evaluate_unseen(
    "Persistence Baseline",
    persistence_predictions,
    time.time() - start
)

del persistence_predictions
gc.collect()

# ------------------------------------------------------------
# 6. LINEAR REGRESSION
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MODEL 2 — LINEAR REGRESSION")
print("=" * 65)

linear_model_unseen = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

start = time.time()

linear_model_unseen.fit(
    X_train_unseen,
    y_train_unseen
)

linear_predictions = linear_model_unseen.predict(
    X_test_unseen
)

linear_time = time.time() - start

evaluate_unseen(
    "Linear Regression",
    linear_predictions,
    linear_time
)

del linear_predictions
gc.collect()

# ------------------------------------------------------------
# 7. RANDOM FOREST
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MODEL 3 — RANDOM FOREST")
print("=" * 65)

rng = np.random.default_rng(RANDOM_STATE)

RF_SAMPLE_SIZE = min(
    250_000,
    len(X_train_unseen)
)

rf_indices = rng.choice(
    len(X_train_unseen),
    size=RF_SAMPLE_SIZE,
    replace=False
)

X_rf = X_train_unseen[rf_indices]
y_rf = y_train_unseen[rf_indices]

rf_model_unseen = RandomForestRegressor(
    n_estimators=60,
    max_depth=16,
    min_samples_leaf=5,
    max_features=0.8,
    max_samples=0.8,
    n_jobs=2,
    random_state=RANDOM_STATE
)

print("Random Forest sample:", len(X_rf))

start = time.time()

rf_model_unseen.fit(X_rf, y_rf)

rf_predictions = rf_model_unseen.predict(
    X_test_unseen
)

rf_time = time.time() - start

evaluate_unseen(
    "Random Forest",
    rf_predictions,
    rf_time
)

del X_rf, y_rf, rf_indices
del rf_predictions, rf_model_unseen
gc.collect()

# ------------------------------------------------------------
# 8. HISTOGRAM GRADIENT BOOSTING
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MODEL 4 — HISTOGRAM GRADIENT BOOSTING")
print("=" * 65)

hgb_model_unseen = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    min_samples_leaf=30,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=15,
    random_state=RANDOM_STATE
)

start = time.time()

hgb_model_unseen.fit(
    X_train_unseen,
    y_train_unseen
)

hgb_predictions = hgb_model_unseen.predict(
    X_test_unseen
)

hgb_time = time.time() - start

evaluate_unseen(
    "HistGradientBoosting",
    hgb_predictions,
    hgb_time
)

del hgb_predictions
gc.collect()

# ------------------------------------------------------------
# 9. FINAL COMPARISON
# ------------------------------------------------------------

unseen_df = pd.DataFrame(
    unseen_results
).sort_values(
    by="MAE"
).reset_index(drop=True)

print("\n" + "=" * 75)
print("UNSEEN-TRAIN MODEL COMPARISON")
print("=" * 75)

print(
    unseen_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 10. SAVE RESULTS
# ------------------------------------------------------------

save_path = os.path.join(
    OUTPUT_DIR,
    "unseen_train_model_comparison.csv"
)

unseen_df.to_csv(
    save_path,
    index=False
)

print("\nResults saved:", save_path)
print("\nUNSEEN-TRAIN EXPERIMENT COMPLETED.")


RAILPULSE AI — UNSEEN-TRAIN EXPERIMENT
Training trains: 2516
Testing trains : 630
Train overlap  : 0

Loading unseen-train split in chunks...
Processed 300,000 rows
Processed 600,000 rows
Processed 900,000 rows

DATA LOADING COMPLETE
Training shape: (870629, 9)
Testing shape : (230814, 9)
Total rows: 1,101,443

MODEL 1 — PERSISTENCE BASELINE

Persistence Baseline
--------------------------------------------------
MAE  : 9.6001
MSE  : 853.1241
RMSE : 29.2083
R2   : 0.8618
Time : 0.00 seconds

MODEL 2 — LINEAR REGRESSION

Linear Regression
--------------------------------------------------
MAE  : 10.4601
MSE  : 831.0544
RMSE : 28.8280
R2   : 0.8654
Time : 0.44 seconds

MODEL 3 — RANDOM FOREST
Random Forest sample: 250000

Random Forest
--------------------------------------------------
MAE  : 11.0897
MSE  : 824.1405
RMSE : 28.7078
R2   : 0.8665
Time : 46.49 seconds

MODEL 4 — HISTOGRAM GRADIENT BOOSTING

HistGradientBoosting
--------------------------------------------------
MAE  : 11.67

In [5]:

# ============================================================
# RAILPULSE AI — ERROR ANALYSIS
# Persistence vs Linear Regression
# ============================================================

import os
import gc
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

OUTPUT_DIR = "../outputs/tables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. PERSISTENCE PREDICTIONS
# ------------------------------------------------------------

delay_index = FEATURE_COLS.index("current_arr_delay")

persistence_pred = X_test_unseen[:, delay_index].astype(np.float64)

# ------------------------------------------------------------
# 2. TRAIN LINEAR REGRESSION AGAIN
# ------------------------------------------------------------

print("Training Linear Regression for error analysis...")

linear_diagnostic = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

linear_diagnostic.fit(
    X_train_unseen,
    y_train_unseen
)

linear_pred = linear_diagnostic.predict(
    X_test_unseen
)

print("Predictions generated.")

# ------------------------------------------------------------
# 3. CREATE DIAGNOSTIC DATAFRAME
# ------------------------------------------------------------

diagnostics = pd.DataFrame({
    "actual_delay": y_test_unseen.astype(np.float64),
    "persistence_prediction": persistence_pred,
    "linear_prediction": linear_pred
})

diagnostics["persistence_error"] = (
    diagnostics["actual_delay"]
    - diagnostics["persistence_prediction"]
)

diagnostics["linear_error"] = (
    diagnostics["actual_delay"]
    - diagnostics["linear_prediction"]
)

diagnostics["persistence_abs_error"] = (
    diagnostics["persistence_error"].abs()
)

diagnostics["linear_abs_error"] = (
    diagnostics["linear_error"].abs()
)

# ------------------------------------------------------------
# 4. GROUP BY ACTUAL DELAY RANGE
# ------------------------------------------------------------

bins = [-0.001, 0, 15, 60, 180, 720, np.inf]

labels = [
    "On time (0)",
    "1–15 min",
    "16–60 min",
    "61–180 min",
    "181–720 min",
    ">720 min"
]

diagnostics["delay_category"] = pd.cut(
    diagnostics["actual_delay"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

category_summary = diagnostics.groupby(
    "delay_category",
    observed=False
).agg(
    observations=("actual_delay", "size"),
    actual_mean_delay=("actual_delay", "mean"),
    persistence_MAE=("persistence_abs_error", "mean"),
    linear_MAE=("linear_abs_error", "mean"),
    persistence_RMSE=(
        "persistence_error",
        lambda x: np.sqrt(np.mean(x ** 2))
    ),
    linear_RMSE=(
        "linear_error",
        lambda x: np.sqrt(np.mean(x ** 2))
    )
).reset_index()

print("\n" + "=" * 80)
print("ERROR ANALYSIS BY ACTUAL NEXT-STATION DELAY")
print("=" * 80)

print(
    category_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

# ------------------------------------------------------------
# 5. OVERALL COMPARISON
# ------------------------------------------------------------

overall = pd.DataFrame([
    {
        "Model": "Persistence",
        "MAE": mean_absolute_error(
            diagnostics["actual_delay"],
            diagnostics["persistence_prediction"]
        ),
        "RMSE": np.sqrt(mean_squared_error(
            diagnostics["actual_delay"],
            diagnostics["persistence_prediction"]
        ))
    },
    {
        "Model": "Linear Regression",
        "MAE": mean_absolute_error(
            diagnostics["actual_delay"],
            diagnostics["linear_prediction"]
        ),
        "RMSE": np.sqrt(mean_squared_error(
            diagnostics["actual_delay"],
            diagnostics["linear_prediction"]
        ))
    }
])

print("\nOVERALL METRICS")
print(overall.to_string(index=False))

# ------------------------------------------------------------
# 6. SAVE TABLES
# ------------------------------------------------------------

category_path = os.path.join(
    OUTPUT_DIR,
    "unseen_train_error_by_delay_category.csv"
)

overall_path = os.path.join(
    OUTPUT_DIR,
    "unseen_train_error_overall.csv"
)

category_summary.to_csv(
    category_path,
    index=False
)

overall.to_csv(
    overall_path,
    index=False
)

print("\nSaved:")
print(category_path)
print(overall_path)

# ------------------------------------------------------------
# 7. LINEAR REGRESSION COEFFICIENTS
# ------------------------------------------------------------

coefficients = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Coefficient": linear_diagnostic.named_steps[
        "model"
    ].coef_
})

coefficients["Absolute_Coefficient"] = (
    coefficients["Coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "Absolute_Coefficient",
    ascending=False
)

print("\nLINEAR REGRESSION COEFFICIENTS")
print(
    coefficients.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

coefficients.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "linear_regression_coefficients.csv"
    ),
    index=False
)

print("\nERROR ANALYSIS COMPLETED.")

# Release model and prediction arrays when done
del linear_diagnostic, linear_pred
del persistence_pred, diagnostics
gc.collect()


Training Linear Regression for error analysis...
Predictions generated.

ERROR ANALYSIS BY ACTUAL NEXT-STATION DELAY
delay_category  observations  actual_mean_delay  persistence_MAE  linear_MAE  persistence_RMSE  linear_RMSE
   On time (0)         80339              0.000            7.383       9.298            32.257       30.867
      1–15 min         62495              7.833            5.778       5.398             9.235        9.026
     16–60 min         58880             31.020           10.077       9.321            15.002       13.969
    61–180 min         20675            100.318           18.557      19.368            32.829       32.387
   181–720 min          7935            319.549           32.215      41.705            79.690       81.958
      >720 min           490            921.327           59.133     101.717           212.193      219.961

OVERALL METRICS
            Model       MAE      RMSE
      Persistence  9.600094 29.208288
Linear Regression 10.460113 28.828

0

In [6]:

# ============================================================
# RAILPULSE AI — DELAY PROPAGATION / DELTA LEARNING
# ============================================================

import os
import gc
import time
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

OUTPUT_DIR = "../outputs/tables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42

# ------------------------------------------------------------
# 1. CREATE DELTA TARGET
# ------------------------------------------------------------

delay_index = FEATURE_COLS.index("current_arr_delay")

current_train_delay = X_train_unseen[:, delay_index]
current_test_delay = X_test_unseen[:, delay_index]

# Target becomes change in delay, not next delay
y_delta_train = (
    y_train_unseen - current_train_delay
).astype(np.float32)

print("Training observations:", len(y_delta_train))
print("Mean delay change:", round(float(y_delta_train.mean()), 3))
print("Minimum delay change:", float(y_delta_train.min()))
print("Maximum delay change:", float(y_delta_train.max()))

# ------------------------------------------------------------
# 2. TRAIN DELTA MODEL
# ------------------------------------------------------------

delta_model = HistGradientBoostingRegressor(
    loss="absolute_error",
    learning_rate=0.06,
    max_iter=100,
    max_leaf_nodes=31,
    min_samples_leaf=40,
    l2_regularization=2.0,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=12,
    random_state=RANDOM_STATE
)

print("\nTraining delay-propagation model...")

start = time.time()

delta_model.fit(
    X_train_unseen,
    y_delta_train
)

training_time = time.time() - start

# ------------------------------------------------------------
# 3. PREDICT DELAY CHANGE
# ------------------------------------------------------------

predicted_delta = delta_model.predict(
    X_test_unseen
)

# Convert predicted change into next-station delay
delta_predictions = (
    current_test_delay + predicted_delta
)

# Arrival delay cannot be negative in our dataset.
# Apply a zero lower bound to predictions.
delta_predictions = np.maximum(
    delta_predictions,
    0
)

# ------------------------------------------------------------
# 4. EVALUATE
# ------------------------------------------------------------

mae = mean_absolute_error(
    y_test_unseen,
    delta_predictions
)

mse = mean_squared_error(
    y_test_unseen,
    delta_predictions
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test_unseen,
    delta_predictions
)

print("\n" + "=" * 65)
print("DELAY PROPAGATION MODEL RESULTS")
print("=" * 65)

print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")
print(f"Time : {training_time:.2f} seconds")

print("\nPersistence MAE : 9.6001")
print("Linear MAE     : 10.4601")

# ------------------------------------------------------------
# 5. SAVE RESULTS
# ------------------------------------------------------------

delta_result = pd.DataFrame([{
    "Model": "HistGradientBoosting Delta Learning",
    "MAE": mae,
    "MSE": mse,
    "RMSE": rmse,
    "R2": r2,
    "Training_Time_Seconds": training_time
}])

delta_path = os.path.join(
    OUTPUT_DIR,
    "unseen_train_delta_model_results.csv"
)

delta_result.to_csv(
    delta_path,
    index=False
)

print("\nSaved:", delta_path)

# ------------------------------------------------------------
# 6. SAVE DELAY-CHANGE DISTRIBUTION
# ------------------------------------------------------------

delta_summary = pd.Series(
    y_delta_train
).describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

print("\nTRAINING DELAY-CHANGE DISTRIBUTION")
print(delta_summary)

print("\nDELTA LEARNING EXPERIMENT COMPLETED.")

# Release temporary arrays
del y_delta_train, predicted_delta
del delta_predictions, delta_model
gc.collect()


Training observations: 870629
Mean delay change: 0.575
Minimum delay change: -4333.0
Maximum delay change: 4322.0

Training delay-propagation model...

DELAY PROPAGATION MODEL RESULTS
MAE  : 9.5035
MSE  : 847.7275
RMSE : 29.1158
R2   : 0.8627
Time : 12.48 seconds

Persistence MAE : 9.6001
Linear MAE     : 10.4601

Saved: ../outputs/tables\unseen_train_delta_model_results.csv

TRAINING DELAY-CHANGE DISTRIBUTION
count    870629.000000
mean          0.574813
std          30.480917
min       -4333.000000
25%          -2.000000
50%           0.000000
75%           4.000000
90%          12.000000
95%          21.000000
99%          55.000000
max        4322.000000
dtype: float64

DELTA LEARNING EXPERIMENT COMPLETED.


29

In [7]:

# ============================================================
# RAILPULSE AI — DELTA MODEL CATEGORY-WISE EVALUATION
# ============================================================

import numpy as np
import pandas as pd
import gc

from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Prepare target
delay_index = FEATURE_COLS.index("current_arr_delay")

current_train = X_train_unseen[:, delay_index]
current_test = X_test_unseen[:, delay_index]

delta_target = (
    y_train_unseen - current_train
).astype(np.float32)

# 2. Train model
print("Training delta model for detailed evaluation...")

delta_eval_model = HistGradientBoostingRegressor(
    loss="absolute_error",
    learning_rate=0.06,
    max_iter=100,
    max_leaf_nodes=31,
    min_samples_leaf=40,
    l2_regularization=2.0,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=12,
    random_state=42
)

delta_eval_model.fit(
    X_train_unseen,
    delta_target
)

# 3. Generate predictions
predicted_change = delta_eval_model.predict(
    X_test_unseen
)

delta_pred = np.maximum(
    current_test + predicted_change,
    0
)

persistence_pred = current_test

# 4. Create evaluation table
error_df = pd.DataFrame({
    "actual": y_test_unseen,
    "persistence_prediction": persistence_pred,
    "delta_prediction": delta_pred
})

error_df["persistence_error"] = (
    error_df["actual"] -
    error_df["persistence_prediction"]
).abs()

error_df["delta_error"] = (
    error_df["actual"] -
    error_df["delta_prediction"]
).abs()

# 5. Categorize by actual delay
bins = [-0.001, 0, 15, 60, 180, 720, np.inf]

labels = [
    "On time (0)",
    "1-15 min",
    "16-60 min",
    "61-180 min",
    "181-720 min",
    ">720 min"
]

error_df["category"] = pd.cut(
    error_df["actual"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# 6. Compare models by category
summary = error_df.groupby(
    "category",
    observed=False
).agg(
    observations=("actual", "size"),
    persistence_MAE=("persistence_error", "mean"),
    delta_MAE=("delta_error", "mean")
).reset_index()

summary["MAE_improvement"] = (
    summary["persistence_MAE"] -
    summary["delta_MAE"]
)

summary["Improvement_Percent"] = (
    summary["MAE_improvement"] /
    summary["persistence_MAE"]
) * 100

print("\n" + "=" * 85)
print("DELTA LEARNING VS PERSISTENCE — BY DELAY CATEGORY")
print("=" * 85)

print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)

# 7. Save results
summary.to_csv(
    "../outputs/tables/delta_category_comparison.csv",
    index=False
)

print("\nSaved: delta_category_comparison.csv")
print("\nPositive MAE improvement means Delta Learning performed better.")

del delta_eval_model, delta_target
del predicted_change, delta_pred, persistence_pred
del error_df
gc.collect()


Training delta model for detailed evaluation...

DELTA LEARNING VS PERSISTENCE — BY DELAY CATEGORY
   category  observations  persistence_MAE  delta_MAE  MAE_improvement  Improvement_Percent
On time (0)         80339            7.383      7.125            0.258                3.493
   1-15 min         62495            5.778      5.426            0.351                6.079
  16-60 min         58880           10.077     10.268           -0.191               -1.892
 61-180 min         20675           18.557     19.051           -0.494               -2.663
181-720 min          7935           32.215     32.087            0.128                0.398
   >720 min           490           59.133     59.027            0.106                0.179

Saved: delta_category_comparison.csv

Positive MAE improvement means Delta Learning performed better.


29